# Data Processing Pipeline

This notebook processes raw JSON interaction logs produced by the [study interface](https://github.com/[repo]) into a structured CSV dataset.

## Setup

Before running, configure the paths and condition mapping in the configuration block below.

**Input — required directory structure:**

```
your-working-directory/
│
├── data/
│   ├── project2-type1/              ← raw JSON files for AI-Single condition
│   │   ├── <timestamp>.json
│   │   ├── <timestamp>.json
│   │   └── ...
│   ├── project2-type2/              ← raw JSON files for Control condition
│   │   └── ...
│   └── project2-type3/              ← raw JSON files for AI-Multiple condition
│       └── ...
│
├── Approved_IDs.csv                 ← participant approval status exported from the crowdsourcing platform
│
└── processing/
    └── data_processing.ipynb        ← this notebook
```

**Output — files generated in `OUTPUT_DIR`:**

```
output/
├── AI-Single.xlsx               ← all AI-Single participants
├── AI-Single_APPROVED.xlsx      ← approved AI-Single participants only
├── Control.xlsx
├── Control_APPROVED.xlsx
├── AI-Multiple.xlsx
└── AI-Multiple_APPROVED.xlsx
```

The subfolder names under `data/` correspond to the study variant identifiers configured in the interface (`projectConfig.ts`). The condition label for each participant is assigned based on the folder name via the `VARIANT_TO_CONDITION` mapping at the top of the notebook. Update `DATA_ROOT`, `APPROVAL_CSV_PATH`, and `OUTPUT_DIR` in the configuration block before running.

## Requirements

```bash
pip install pandas openpyxl
```

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from collections import Counter

# ============================================================
# CONFIGURATION — update these before running
# ============================================================

# Root folder containing one subfolder per study variant
DATA_ROOT = "/path/to/data"  # <-- update this

# Map each variant folder name to a condition label
# These folder names match the study variant names in the interface (projectConfig.ts)
VARIANT_TO_CONDITION = {
    "project2-type1": "AI-Single",
    "project2-type2": "Control",
    "project2-type3": "AI-Multiple",
}

# Path to worker id approval CSV
# Expected columns: 'Participant id', 'Status'
APPROVAL_CSV_PATH = "/path/to/Approved_IDs.csv"  # <-- update this

# Output directory for the processed datasets (one file per condition)
OUTPUT_DIR = "/path/to/output"  # <-- update this

# ============================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)


# Page 1 - Consent
def process_page1(data):
    page = data.get("page1", {})
    value = page.get("value", "")
    time = page.get("time", None)
    return {
        "consent_given": value == "I agree to participate in the research.",
        "page1_submission_time": time
    }


# Page 2 - Participant ID
def process_page2(data):
    page = data.get("page2", {})
    value = page.get("value", None)
    time = page.get("time", None)
    return {
        "participant_id": value,
        "page2_submission_time": time
    }


# Page 3 - Instruction Check
def process_page3(data):
    page3 = data.get("page3", {})
    submission_time = page3.get("time", None)
    return {
        "instruction_checks_passed": submission_time is not None,
        "page3_submission_time": submission_time
    }


# Page 8 - Demographics
# Fields extracted by index position (order is fixed across all conditions):
# 0: llm_familiarity, 1: llm_use_frequency, 2: llm_attitude, 3: age,
# 4: education, 5: major, 6: gender, 7: race,
# 8: everyday_poem_reading, 9: everyday_poem_enjoyment,
# 10: humanities_course_experience, 11: poetry_course_experience
def process_page8(data):
    page8 = data.get("page8", {})
    info = page8.get("info", [])

    index_map = [
        "llm_familiarity",
        "llm_use_frequency",
        "llm_attitude",
        "age",
        "education",
        "major",
        "gender",
        "race",
        "everyday_poem_reading",
        "everyday_poem_enjoyment",
        "humanities_course_experience",
        "poetry_course_experience",
    ]

    result = {}
    for i, key in enumerate(index_map):
        answer = None
        if i < len(info):
            answer = info[i].get("answer") if "answer" in info[i] else info[i].get("value")
        if key == "major" and isinstance(answer, list):
            result[key] = "; ".join(answer)
        else:
            result[key] = answer

    result["page8_submission_time"] = page8.get("time")
    return result


# Identity alignment variables
def add_identity_alignment_variables(row):
    gender = str(row.get("gender", "")).strip()
    race = str(row.get("race", "")).strip()

    love_poem_author   = {"gender": "Female", "race": "White"}
    dusting_author     = {"gender": "Female", "race": "Black or African American"}
    english_b_author   = {"gender": "Male",   "race": "Black or African American"}

    row["love_poem_gender_match"]  = int(gender == love_poem_author["gender"])
    row["dusting_gender_match"]    = int(gender == dusting_author["gender"])
    row["english_b_gender_match"]  = int(gender == english_b_author["gender"])
    row["love_poem_race_match"]    = int(race == love_poem_author["race"])
    row["dusting_race_match"]      = int(race == dusting_author["race"])
    row["english_b_race_match"]    = int(race == english_b_author["race"])

    return row


# Poem order
def process_poem_order(data):
    suffix_to_poem = {"1": "love_poem", "2": "dusting", "3": "english_b"}
    timestamped = []
    for suffix in ["1", "2", "3"]:
        key = f"page4_{suffix}"
        if key in data and isinstance(data[key], dict):
            t = data[key].get("time")
            if t:
                timestamped.append((t, suffix_to_poem[suffix]))
    timestamped.sort(key=lambda x: x[0])
    poem_order = [poem for _, poem in timestamped]
    return {"poem_order": ";".join(poem_order) if poem_order else None}


# Helper: hover duration from enter/leave events
def compute_hover_duration(events):
    total = 0
    enter_stack = []
    for e in events:
        if e.get("type") == "enter":
            enter_stack.append(e.get("time"))
        elif e.get("type") == "leave" and enter_stack:
            enter_time = enter_stack.pop(0)
            leave_time = e.get("time")
            if enter_time is not None and leave_time is not None:
                total += leave_time - enter_time
    return total


# Helper: format line hover indices
def format_hover_index_string(index_list):
    count = Counter(index_list)
    formatted = []
    for idx in sorted(count.keys()):
        n = count[idx]
        formatted.append(f"{idx}" if n == 1 else f"{idx}({n})")
    return ", ".join(formatted)


# Pages 4_X - First Reading
def process_page4(data):
    result = {}
    suffix_to_poem = {"1": "love_poem", "2": "dusting", "3": "english_b"}

    for suffix, poem_name in suffix_to_poem.items():
        key = f"page4_{suffix}"
        page = data.get(key, {})

        hover_lines = [entry.get("index") for entry in page.get("poemItemHover", []) if "index" in entry]
        result[f"{poem_name}_first_reading_line_hover"] = format_hover_index_string(hover_lines) if hover_lines else None

        left_events = page.get("left_content_hover", [])
        result[f"{poem_name}_reading_instruction_hover_duration"] = compute_hover_duration(left_events) if left_events else 0

        right_events = page.get("right_content_hover", [])
        result[f"{poem_name}_first_reading_hover_duration"] = compute_hover_duration(right_events) if right_events else 0

        result[f"page4_{suffix}_submission_time"] = page.get("time", None)

    return result


# Pages 5_X - Interpretation Tasks
def process_page5(data):
    textbox_map = {
        0: "first_example",
        1: "first_explanation",
        2: "second_example",
        3: "second_explanation",
        4: "third_example",
        5: "third_explanation"
    }
    suffix_to_poem = {"1": "love_poem", "2": "dusting", "3": "english_b"}

    def compute_duration(events):
        total = 0
        stack = []
        for e in events:
            if e["type"] == "enter":
                stack.append(e["time"])
            elif e["type"] == "leave" and stack:
                total += e["time"] - stack.pop(0)
        return total

    def format_line_hover(hover_data):
        counter = Counter()
        for item in hover_data:
            idx = item.get("index")
            if idx is not None:
                counter[idx] += 1
        return ", ".join([f"{i}" if c == 1 else f"{i}({c})" for i, c in sorted(counter.items())])

    row = {}

    for suffix, poem_name in suffix_to_poem.items():
        key = f"page5_{suffix}"
        page = data.get(key, {})

        # --- Final Answers ---
        for idx, label in textbox_map.items():
            answer = ""
            if idx < len(page.get("info", [])):
                answer = page["info"][idx].get("value", "")
            row[f"{poem_name}_{label}_answer"] = answer

        # --- Copied Text ---
        for idx, label in textbox_map.items():
            copies = page.get("copy_text", {}).get(str(idx), {}).get("copy_text", [])
            row[f"{poem_name}_{label}_copied_text"] = " | ".join(copies) if copies else ""

        # --- Input and Pause Behavior ---
        input_sequence = []
        for idx, label in textbox_map.items():
            input_events = page.get(f"input_time_{idx}", [])
            starts = [e["time"] for e in input_events if e.get("type") == "start"]
            ends   = [e["time"] for e in input_events if e.get("type") == "end"]
            durations = [end - start for start, end in zip(starts, ends) if end > start]

            row[f"{poem_name}_{label}_num_input"] = len(durations)
            row[f"{poem_name}_{label}_input_duration"] = page.get(f"input_total_time_{idx}", 0)

            if starts:
                input_sequence.append((min(starts), idx))

            stop_events = page.get(f"stop_input_time_{idx}", [])
            stop_starts = [e["time"] for e in stop_events if e.get("type") == "start"]
            stop_ends   = [e["time"] for e in stop_events if e.get("type") == "end"]
            stop_durations = [end - start for start, end in zip(stop_starts, stop_ends) if end > start]

            row[f"{poem_name}_{label}_num_pause"] = len(stop_durations)
            row[f"{poem_name}_{label}_pause_duration"] = sum(stop_durations) if stop_durations else 0

        row[f"{poem_name}_input_sequence"] = (
            "|".join(str(idx) for _, idx in sorted(input_sequence)) if input_sequence else ""
        )

        # --- AI Text Selection ---
        ai_texts = page.get("ai_select_text", [])
        row[f"{poem_name}_ai_select_text"] = " | ".join(ai_texts) if ai_texts else ""

        # --- AI Tab Viewing ---
        for i in range(1, 4):
            tap_events = page.get(f"tap{i}", [])
            tap_starts = [e.get("time") for e in tap_events if e.get("type") == "start"]
            tap_ends   = [e.get("time") for e in tap_events if e.get("type") == "end"]
            view_count = min(len(tap_starts), len(tap_ends))
            row[f"{poem_name}_ai_tab{i}_num_view"] = view_count
            row[f"{poem_name}_ai_tab{i}_duration"] = page.get(f"tab_total_time{i}", 0)

        # --- AI Hide Duration (pre-computed) ---
        row[f"{poem_name}_ai_hide_duration"] = page.get("hide_total_time", 0)

        # --- Poem Line Hover ---
        row[f"{poem_name}_interpreting_line_hover"] = format_line_hover(page.get("poemItemHover", []))

        # --- Panel Hover Counts and Durations ---
        hover_areas = {
            "question":    "question_hover",
            "ai":          "ai_hover",
            "poem":        "poem_hover",
            "answer_area": "answer_hover",
            "rest_area":   "rest_area"
        }
        hover_total_duration = 0
        for area_label, json_key in hover_areas.items():
            events = page.get(json_key, [])
            enter_count = sum(1 for e in events if e.get("type") == "enter")
            duration = compute_duration(events)
            row[f"{poem_name}_{area_label}_num_entries"] = enter_count
            row[f"{poem_name}_{area_label}_hover_total_duration"] = duration
            hover_total_duration += duration
        row[f"{poem_name}_hover_total_duration"] = hover_total_duration

        # --- Pre-Computed Total Durations ---
        row[f"{poem_name}_question_total_duration"]     = page.get("question_total", 0)
        row[f"{poem_name}_ai_total_duration"]           = page.get("ai_total", 0)
        row[f"{poem_name}_poem_total_duration"]         = page.get("poem_total", 0)
        row[f"{poem_name}_answer_area_total_duration"]  = page.get("answer_total", 0)
        row[f"{poem_name}_rest_area_total_duration"]    = page.get("rest_area_total_time", 0)
        row[f"{poem_name}_cursor_activity_total_duration"] = sum([
            page.get("question_total", 0),
            page.get("ai_total", 0),
            page.get("poem_total", 0),
            page.get("answer_total", 0),
            page.get("rest_area_total_time", 0)
        ])

        # --- Submission Time ---
        row[f"page5_{suffix}_submission_time"] = page.get("time", None)

    return row


# Pages 6_X - Post-Task Ratings
def process_page6(data):
    suffix_to_poem = {"1": "love_poem", "2": "dusting", "3": "english_b"}

    question_fields = [
        "close_reading_difficulty",
        "close_reading_appreciation",
        "close_reading_appreciation_rationale",
        "close_reading_enjoyment",
        "close_reading_enjoyment_rationale",
        "close_reading_self_efficacy",
        "close_reading_self_efficacy_rationale",
        "close_reading_ai_use",
        "close_reading_ai_helpfulness"
    ]

    row = {}

    for suffix, poem_name in suffix_to_poem.items():
        key = f"page6_{suffix}"
        page = data.get(key, {})
        info = page.get("info", [])

        for i, field_suffix in enumerate(question_fields):
            answer = ""
            if i < len(info):
                answer = info[i].get("answer", "").strip()
            row[f"{poem_name}_{field_suffix}"] = answer

        row[f"page6_{suffix}_submission_time"] = page.get("time", None)

    return row


# Page 7 - Debrief and Study Feedback
def process_page7(data):
    page = data.get("page7", {})
    info = page.get("info", [])
    result = {}

    if len(info) >= 1:
        answer_0 = info[0].get("answer")
        result["poem_read_prior_participation"] = "; ".join(answer_0) if isinstance(answer_0, list) else answer_0
    if len(info) >= 2:
        result["task_approach"] = info[1].get("answer")
    if len(info) >= 3:
        result["study_feedback"] = info[2].get("answer")

    result["page7_submission_time"] = page.get("time")
    return result


# Page leave tracking
def process_leave_page_logs(data):
    row = {}
    expected_pages = (
        [f"page{i}" for i in range(1, 10)] +
        [f"page4_{i}" for i in range(1, 4)] +
        [f"page5_{i}" for i in range(1, 4)] +
        [f"page6_{i}" for i in range(1, 4)]
    )

    for key in expected_pages:
        page_data = data.get(key, {})
        leave_events = page_data.get("leave_page", [])

        leave_times  = [e["time"] for e in leave_events if e.get("type") == "leave"]
        return_times = [e["time"] for e in leave_events if e.get("type") == "return"]
        pairs = min(len(leave_times), len(return_times))

        duration = sum(
            return_times[i] - leave_times[i]
            for i in range(pairs)
            if leave_times[i] is not None and return_times[i] is not None
        )

        row[f"leaved_{key}_times"]    = pairs
        row[f"leaved_{key}_duration"] = duration

    return row


# === Main processor: one folder per condition ===
def process_folder(folder_path, condition_label):
    rows = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r") as f:
                try:
                    data = json.load(f)
                    row = {"filename": filename, "condition": condition_label}
                    row.update(process_page1(data))
                    row.update(process_page2(data))
                    row.update(process_page3(data))
                    row.update(process_page8(data))
                    row = add_identity_alignment_variables(row)
                    row.update(process_poem_order(data))
                    row.update(process_page4(data))
                    row.update(process_page5(data))
                    row.update(process_page6(data))
                    row.update(process_page7(data))
                    row.update(process_leave_page_logs(data))
                    row["is_refresh"] = data.get("isRefresh", False)
                    rows.append(row)
                except Exception as e:
                    print(f"\u274c Error processing {filename}: {e}")
    return rows


# === Load approved participant IDs ===
approved_df  = pd.read_csv(APPROVAL_CSV_PATH)
approved_ids = approved_df.loc[approved_df["Status"] == "APPROVED", "Participant id"].dropna().astype(str).unique()


# === Run across all variant folders — one output file per condition ===
for variant_folder, condition_label in VARIANT_TO_CONDITION.items():
    folder_path = os.path.join(DATA_ROOT, variant_folder)
    if not os.path.isdir(folder_path):
        print(f"\u26a0\ufe0f  Folder not found, skipping: {folder_path}")
        continue

    rows = process_folder(folder_path, condition_label)
    print(f"\u2705 {condition_label}: {len(rows)} participants processed from '{variant_folder}'")

    df = pd.DataFrame(rows)

    # Save all participants
    output_path = os.path.join(OUTPUT_DIR, f"{condition_label}.xlsx")
    df.to_excel(output_path, index=False)
    print(f"   Saved: {output_path}")

    # Save approved participants only
    df_approved = df[df["participant_id"].isin(approved_ids)].copy()
    output_approved_path = os.path.join(OUTPUT_DIR, f"{condition_label}_APPROVED.xlsx")
    df_approved.to_excel(output_approved_path, index=False)
    print(f"   Approved ({len(df_approved)} rows): {output_approved_path}")

print("\n\u2705 Done.")